### 3rd august, 2026

With NLR sequences from TSL from NLRtracker, extract the necesary ones to yse to run phylogeny.

From the NLRtracker results tsv, add a bed file, then we cn extract the one we can use fomr the losu

In [1]:
import pandas as pd
from pathlib import Path
import os

In [2]:
workDir="/Users/waweru/Documents/2022/IBSpy_Targets/2024-12-05_5B_Yellow_Rust/2025-07-02_Watkins_assemblies/NLRTrackerTSL"
bedFiles=f"{workDir}/bedFiles"
nlrTrackerFiles=f"{workDir}/NLRtracker"

In [4]:
os.chdir(workDir)
os.getcwd()

'/Users/waweru/Documents/2022/IBSpy_Targets/2024-12-05_5B_Yellow_Rust/2025-07-02_Watkins_assemblies/NLRTrackerTSL'

In [33]:
genomes = ["arinaLrFor",
"chinesespring",
"julius",
"jagger",
"lancer",
"landmark",
"norin61",
"mace",
"stanley",
"symattis",
"cadenza_v2",
"fielder_v1",
"paragon_v3",
"renan_v2.1",
"kariega_v1",
"WATDE0042",
"WATDE0100"]

gene names in bed file are in small caps, whicle in all CAPS in the nlrtracker output file

In [ ]:
# this one accounts for the `.1` and the `.` in V2.1 being X in the NLRTracker file,

In [34]:
import re
import pandas as pd


def normalise_gene_id(gene_id):
    """
    Standardise BED and NLRTracker gene IDs for matching.

    Examples
    --------
    kariega_v1_TraesKAR5B01G0004900.1
        -> KARIEGA_V1_TRAESKAR5B01G0004900

    renan_v2.1_TraesRN5B0100011400
        -> RENAN_V2X1_TRAESRN5B0100011400

    RENAN_V2X1_TRAESRN5B0100011400
        -> RENAN_V2X1_TRAESRN5B0100011400
    """

    if pd.isna(gene_id):
        return ""

    gene_id = str(gene_id).strip().upper()

    # Remove a terminal transcript suffix such as .1, .2 or .10.
    # This must happen before replacing remaining dots with X.
    gene_id = re.sub(r"\.\d+$", "", gene_id)

    # NLRTracker replaces dots in genome names with X.
    # For example: RENAN_V2.1 -> RENAN_V2X1
    gene_id = gene_id.replace(".", "X")

    return gene_id

In [44]:
workDir="/Users/waweru/Documents/2022/IBSpy_Targets/2024-12-05_5B_Yellow_Rust/2025-07-02_Watkins_assemblies/NLRTrackerTSL"
bedDir=f"{workDir}/bedFiles"
nlrTrackerDir=f"{workDir}/NLRtracker"
outputDir=f"{workDir}/NLRTrackerDomainFiles+geneCoords"

os.makedirs(outputDir, exist_ok=True)

In [45]:
# --------------------------------------------------------------
# Process each genome
# --------------------------------------------------------------

summary = []

for genome in genomes:

    print(f"\nProcessing: {genome}")

    bedFile = f"{bedDir}/{genome}_chr5B_0-100Mb.bed"

    nlrTrackerTsv = f"{nlrTrackerDir}/nlrtracker_{genome}_chr5B_0-100Mb_longestIsoform.fa/nlrtracker_{genome}_chr5B_0-100Mb_longestIsoform.fa_Domains.tsv"

    outputFile = f"{outputDir}/nlrtracker_{genome}_chr5B_0-100Mb_with_coordinates.tsv"

    unmatchedFile = f"{outputDir}/nlrtracker_{genome}_chr5B_0-100Mb_unmatched_genes.tsv"

    # ----------------------------------------------------------
    # Check that both input files exist
    # ----------------------------------------------------------

    if not os.path.exists(bedFile):
        print(f"BED file not found: {bedFile}")
        continue

    if not os.path.exists(nlrTrackerTsv):
        print(f"NLRTracker file not found: {nlrTrackerTsv}")
        continue
    # ----------------------------------------------------------
    # Read BED file
    # ----------------------------------------------------------

    bed = pd.read_csv(
        bedFile,
        sep="\t",
        header=None,
        comment="#",
        names=[
            "chromosome",
            "gene_start",
            "gene_end",
            "bed_gene_id",
            "bed_score",
            "gene_strand",
        ],
        dtype={
            "chromosome": str,
            "gene_start": int,
            "gene_end": int,
            "bed_gene_id": str,
            "bed_score": str,
            "gene_strand": str,
        },
    )

        # Remove LC genes.
    bed = bed.loc[
        ~bed["bed_gene_id"].str.contains(
            "LC",
            case=False,
            na=False,
        )
    ].copy()

        # Standardise BED gene names.
    bed["gene_key"] = bed["bed_gene_id"].apply(
        normalise_gene_id
    )

    # Check for duplicated gene IDs in the BED file.
    duplicatedBedGenes = bed.loc[
        bed["gene_key"].duplicated(keep=False)
    ].copy()

    if not duplicatedBedGenes.empty:
        duplicateFile = (
            outputDir
            / f"{genome}_duplicated_bed_genes.tsv"
        )

        duplicatedBedGenes.to_csv(
            duplicateFile,
            sep="\t",
            index=False,
        )

        print(
            f"Warning: duplicated BED gene IDs found. "
            f"Saved to {duplicateFile}"
        )

    # Keep one coordinate record per gene.
    bedCoordinates = (
        bed[
            [
                "gene_key",
                "chromosome",
                "gene_start",
                "gene_end",
                "gene_strand",
            ]
        ]
        .drop_duplicates(subset="gene_key")
        .copy()
    )

    # ----------------------------------------------------------
    # Read NLRTracker output
    # ----------------------------------------------------------

    nlrTracker = pd.read_csv(
        nlrTrackerTsv,
        sep="\t",
        dtype=str,
        keep_default_na=False,
    )

    if "seqname" not in nlrTracker.columns:
        print(
            f"'seqname' column not found in: "
            f"{nlrTrackerTsv}"
        )
        continue

    # Standardise NLRTracker gene IDs for matching.
    nlrTracker["gene_key"] = (
        nlrTracker["seqname"]
        .apply(normalise_gene_id)
    )

    # ----------------------------------------------------------
    # Merge BED coordinates into NLRTracker table
    # ----------------------------------------------------------
    # The same gene may occur on several NLRTracker rows because
    # CC, NBARC and LRR domains are listed separately.

    merged = nlrTracker.merge(
        bedCoordinates,
        on="gene_key",
        how="left",
        validate="many_to_one",
    )

        # ----------------------------------------------------------
    # Calculate gene and domain lengths
    # ----------------------------------------------------------

    # Convert coordinate columns to numeric values.
    merged["gene_start"] = pd.to_numeric(
        merged["gene_start"],
        errors="coerce",
    )

    merged["gene_end"] = pd.to_numeric(
        merged["gene_end"],
        errors="coerce",
    )

    merged["start"] = pd.to_numeric(
        merged["start"],
        errors="coerce",
    )

    merged["end"] = pd.to_numeric(
        merged["end"],
        errors="coerce",
    )

    # BED coordinates are 0-based and end-exclusive.
    merged["gene_length_bp"] = (
        merged["gene_end"] - merged["gene_start"]
    )

    # NLRTracker protein-domain coordinates are treated as inclusive.
    merged["domain_length_aa"] = (
        merged["end"] - merged["start"] + 1
    )

    # Calculate length directly from the reported domain sequence.
    merged["sequence_length_aa"] = (
        merged["sequence"]
        .fillna("")
        .str.replace(r"\s+", "", regex=True)
        .str.len()
    )

    # ----------------------------------------------------------
    # Reorder columns
    # ----------------------------------------------------------
    originalColumns = [
        column
        for column in nlrTracker.columns
        if column != "gene_key"
    ]

    coordinateColumns = [
        "chromosome",
        "gene_start",
        "gene_end",
        "gene_length_bp",
        "gene_strand",
    ]

    # These will be placed after the NLRTracker domain end coordinate.
    domainLengthColumns = [
        "domain_length_aa",
        "sequence_length_aa",
    ]

    seqnamePosition = originalColumns.index("seqname") + 1

    outputColumns = (
        originalColumns[:seqnamePosition]
        + coordinateColumns
        + originalColumns[seqnamePosition:]
    )

    # Place domain-length columns immediately after the original
    # NLRTracker 'end' column.
    domainEndPosition = outputColumns.index("end") + 1

    outputColumns = (
        outputColumns[:domainEndPosition]
        + domainLengthColumns
        + outputColumns[domainEndPosition:]
    )

    merged = merged.drop(columns="gene_key")
    merged = merged[outputColumns]

    # ----------------------------------------------------------
    # Find genes that did not match the BED file
    # ----------------------------------------------------------

    unmatchedGenes = (
        merged.loc[
            merged["chromosome"].isna(),
            ["seqname"],
        ]
        .drop_duplicates()
        .sort_values("seqname")
    )

    matchedGenes = (
        merged.loc[
            merged["chromosome"].notna(),
            "seqname",
        ]
        .nunique()
    )

    numberUnmatchedGenes = len(unmatchedGenes)

    # ----------------------------------------------------------
    # Save output
    # ----------------------------------------------------------

    merged.to_csv(
        outputFile,
        sep="\t",
        index=False,
        na_rep="NA",
    )

    if numberUnmatchedGenes > 0:
        unmatchedGenes.to_csv(
            unmatchedFile,
            sep="\t",
            index=False,
        )

    print(f"Matched genes:   {matchedGenes}")
    print(f"Unmatched genes: {numberUnmatchedGenes}")
    print(f"Output:          {outputFile}")

    summary.append(
        {
            "genome": genome,
            "total_nlrtracker_rows": len(merged),
            "matched_rows": merged["chromosome"].notna().sum(),
            "unmatched_rows": merged["chromosome"].isna().sum(),
            "matched_genes": matchedGenes,
            "unmatched_genes": numberUnmatchedGenes,
            "output_file": str(outputFile),
        }
    )


# --------------------------------------------------------------
# Save processing summary
# --------------------------------------------------------------

summaryDf = pd.DataFrame(summary)

summaryFile = f"{outputDir}/NLRTracker_coordinate_matching_summary.tsv"

summaryDf.to_csv(
    summaryFile,
    sep="\t",
    index=False,
)

print(f"\nSummary written to: {summaryFile}")


Processing: arinaLrFor
Matched genes:   32
Unmatched genes: 0
Output:          /Users/waweru/Documents/2022/IBSpy_Targets/2024-12-05_5B_Yellow_Rust/2025-07-02_Watkins_assemblies/NLRTrackerTSL/NLRTrackerDomainFiles+geneCoords/nlrtracker_arinaLrFor_chr5B_0-100Mb_with_coordinates.tsv

Processing: chinesespring
Matched genes:   38
Unmatched genes: 0
Output:          /Users/waweru/Documents/2022/IBSpy_Targets/2024-12-05_5B_Yellow_Rust/2025-07-02_Watkins_assemblies/NLRTrackerTSL/NLRTrackerDomainFiles+geneCoords/nlrtracker_chinesespring_chr5B_0-100Mb_with_coordinates.tsv

Processing: julius
Matched genes:   29
Unmatched genes: 0
Output:          /Users/waweru/Documents/2022/IBSpy_Targets/2024-12-05_5B_Yellow_Rust/2025-07-02_Watkins_assemblies/NLRTrackerTSL/NLRTrackerDomainFiles+geneCoords/nlrtracker_julius_chr5B_0-100Mb_with_coordinates.tsv

Processing: jagger
Matched genes:   32
Unmatched genes: 0
Output:          /Users/waweru/Documents/2022/IBSpy_Targets/2024-12-05_5B_Yellow_Rust/2025-07-

In [37]:
nlrIntervalCoords = f"{workDir}/cultivarsNLRGeneCoordinates_17PGsLoc1.txt"

In [46]:
import pandas as pd
from pathlib import Path


# --------------------------------------------------------------
# Input and output paths
# --------------------------------------------------------------

intervalFile = f"{workDir}/cultivarsNLRGeneCoordinates_17PGsLoc1.txt"

nlrCoordinateDir = f"{workDir}/NLRTrackerDomainFiles+geneCoords"

outputDir = f"{workDir}/NBARCSequencesWithinLOC1Intervals"

os.makedirs(outputDir, exist_ok=True)


# --------------------------------------------------------------
# Read the cultivar-specific intervals
# --------------------------------------------------------------
# The file is whitespace-separated:
#
# cultivar chromosome start end
# arinaLrFor chr5B 4806139 10761405
# landmark   chr5B 1975397 8497693

intervals = pd.read_csv(
    intervalFile,
    sep=r"\s+",
    dtype={
        "cultivar": str,
        "chromosome": str,
        "start": int,
        "end": int,
    },
)

# Remove duplicated interval rows, such as the repeated arinaLrFor row.
intervals = intervals.drop_duplicates(
    subset=["cultivar", "chromosome", "start", "end"]
)

print(f"Number of intervals: {len(intervals)}")


# --------------------------------------------------------------
# Store results from all genomes
# --------------------------------------------------------------

allNbarcRows = []
fastaRecords = []
summary = []


# --------------------------------------------------------------
# Process each cultivar interval
# --------------------------------------------------------------

for _, interval in intervals.iterrows():

    genome = interval["cultivar"]
    chromosome = interval["chromosome"]
    intervalStart = interval["start"]
    intervalEnd = interval["end"]

    print(f"\nProcessing: {genome}")
    print(
        f"Interval: {chromosome}:"
        f"{intervalStart}-{intervalEnd}"
    )

    nlrTrackerFile = f"{nlrCoordinateDir}/nlrtracker_{genome}_chr5B_0-100Mb_with_coordinates.tsv"

    if not os.path.exists(nlrTrackerFile):
        print(f"File not found: {nlrTrackerFile}")
        continue

    # ----------------------------------------------------------
    # Read the coordinate-annotated NLRTracker table
    # ----------------------------------------------------------

    nlrTracker = pd.read_csv(
        nlrTrackerFile,
        sep="\t",
        dtype=str,
        keep_default_na=False,
    )

    requiredColumns = {
        "seqname",
        "chromosome",
        "gene_start",
        "gene_end",
        "description",
        "sequence",
    }

    missingColumns = requiredColumns.difference(
        nlrTracker.columns
    )

    if missingColumns:
        print(
            f"Skipping {genome}; missing columns: "
            f"{sorted(missingColumns)}"
        )
        continue

    # Convert genomic coordinates to numbers.
    nlrTracker["gene_start"] = pd.to_numeric(
        nlrTracker["gene_start"],
        errors="coerce",
    )

    nlrTracker["gene_end"] = pd.to_numeric(
        nlrTracker["gene_end"],
        errors="coerce",
    )

    # Remove rows lacking genomic coordinates.
    nlrTracker = nlrTracker.dropna(
        subset=["gene_start", "gene_end"]
    ).copy()

    nlrTracker["gene_start"] = (
        nlrTracker["gene_start"].astype(int)
    )

    nlrTracker["gene_end"] = (
        nlrTracker["gene_end"].astype(int)
    )

    # ----------------------------------------------------------
    # Select NBARC domain rows
    # ----------------------------------------------------------

    isNbarc = (
        nlrTracker["description"]
        .str.strip()
        .str.upper()
        .eq("NBARC")
    )

    # Chromosome comparison is made case-insensitive.
    isCorrectChromosome = (
        nlrTracker["chromosome"]
        .str.strip()
        .str.upper()
        .eq(chromosome.upper())
    )

    # ----------------------------------------------------------
    # Select genes overlapping the requested interval
    # ----------------------------------------------------------
    # A gene overlaps the interval when:
    #
    # gene_end >= interval_start
    # AND
    # gene_start <= interval_end

    withinInterval = (
        (nlrTracker["gene_start"] >= intervalStart)
        &
        (nlrTracker["gene_end"] <= intervalEnd)
    )

    selected = nlrTracker.loc[
        isNbarc
        & isCorrectChromosome
        & withinInterval
    ].copy()

    # Add interval and cultivar information.
    selected.insert(0, "cultivar", genome)

    selected["interval_chromosome"] = chromosome
    selected["interval_start"] = intervalStart
    selected["interval_end"] = intervalEnd

    # ----------------------------------------------------------
    # Save a separate filtered table for each cultivar
    # ----------------------------------------------------------

    cultivarOutput = f"{outputDir}/{genome}_NBARC_within_interval.tsv"

    selected.to_csv(
        cultivarOutput,
        sep="\t",
        index=False,
    )

    print(f"NBARC sequences found: {len(selected)}")
    print(f"Output: {cultivarOutput}")

    allNbarcRows.append(selected)

    # ----------------------------------------------------------
    # Create FASTA records
    # ----------------------------------------------------------

    for _, row in selected.iterrows():

        # The NLRTracker start and end columns represent
        # domain coordinates within the protein.
        domainStart = row.get("start", "")
        domainEnd = row.get("end", "")

        fastaHeader = (
            f">{genome}|{row['seqname']}|"
            f"{row['chromosome']}:"
            f"{row['gene_start']}-{row['gene_end']}|"
            f"NBARC:{domainStart}-{domainEnd}"
        )

        fastaSequence = row["sequence"].strip()

        if fastaSequence:
            fastaRecords.append(
                f"{fastaHeader}\n{fastaSequence}"
            )

    summary.append(
        {
            "cultivar": genome,
            "chromosome": chromosome,
            "interval_start": intervalStart,
            "interval_end": intervalEnd,
            "nbarc_sequences": len(selected),
            "input_file": str(nlrTrackerFile),
            "output_file": str(cultivarOutput),
        }
    )


# --------------------------------------------------------------
# Save one combined TSV file
# --------------------------------------------------------------

if allNbarcRows:

    combinedNbarc = pd.concat(
        allNbarcRows,
        ignore_index=True,
    )

    combinedTsv = f"{outputDir}/all17Genomes_NBARCsWithinLOC1Intervals.tsv"

    combinedNbarc.to_csv(
        combinedTsv,
        sep="\t",
        index=False,
    )

    print(f"\nCombined TSV: {combinedTsv}")

else:
    print("\nNo NBARC rows were found.")


# --------------------------------------------------------------
# Save one combined FASTA file
# --------------------------------------------------------------

combinedFasta = f"{outputDir}/all17Genomes_NBARCsWithinLOC1Intervals.fa"

with open(combinedFasta,"w") as fastaHandle:

    if fastaRecords:
        fastaHandle.write(
            "\n".join(fastaRecords) + "\n"
        )

print(f"Combined FASTA: {combinedFasta}")


# --------------------------------------------------------------
# Save a summary
# --------------------------------------------------------------

summaryDf = pd.DataFrame(summary)

summaryFile = f"{outputDir}/NBARC_interval_extraction_summary.tsv"

summaryDf.to_csv(
    summaryFile,
    sep="\t",
    index=False,
)

print(f"Summary: {summaryFile}")

Number of intervals: 17

Processing: arinaLrFor
Interval: chr5B:4806139-10761405
NBARC sequences found: 0
Output: /Users/waweru/Documents/2022/IBSpy_Targets/2024-12-05_5B_Yellow_Rust/2025-07-02_Watkins_assemblies/NLRTrackerTSL/NBARCSequencesWithinLOC1Intervals/arinaLrFor_NBARC_within_interval.tsv

Processing: landmark
Interval: chr5B:1975397-8497693
NBARC sequences found: 0
Output: /Users/waweru/Documents/2022/IBSpy_Targets/2024-12-05_5B_Yellow_Rust/2025-07-02_Watkins_assemblies/NLRTrackerTSL/NBARCSequencesWithinLOC1Intervals/landmark_NBARC_within_interval.tsv

Processing: julius
Interval: chr5B:8888822-13102316
NBARC sequences found: 6
Output: /Users/waweru/Documents/2022/IBSpy_Targets/2024-12-05_5B_Yellow_Rust/2025-07-02_Watkins_assemblies/NLRTrackerTSL/NBARCSequencesWithinLOC1Intervals/julius_NBARC_within_interval.tsv

Processing: norin61
Interval: chr5B:8404244-11819510
NBARC sequences found: 5
Output: /Users/waweru/Documents/2022/IBSpy_Targets/2024-12-05_5B_Yellow_Rust/2025-07-02_